# IISc VCL Assignment — Tracking-by-Detection on VisDrone (Unique Pipeline)

This notebook is a **from-scratch, modular, non-proprietary** pipeline for the IISc Visual Computing Lab assignment.

## What is implemented
1. **Detection stage** (VisDrone-DET train/val)
   - Convert VisDrone DET annotations to COCO JSON
   - Fine-tune a torchvision detector (**RetinaNet**) on train split
   - Evaluate on val split with `pycocotools`
   - Print: Precision, Recall, mAP@0.50, mAP@0.50:0.95, mAR@0.50, mAR@0.50:0.95

2. **Tracking stage** (VisDrone-MOT val)
   - Run detector frame-by-frame
   - Track with a **ByteTrack-style two-threshold IoU association tracker** (custom `ByteLiteTracker`)
   - Evaluate with `motmetrics` and `TrackEval`
   - Print: HOTA, MOTA, IDSW, Precision, Recall

> Hard deadline reference in the assignment mail: **Friday, April 17, 2026, 23:59**.


In [ ]:
# =============================
# 0) Environment setup
# =============================
!nvidia-smi
!pip -q install pycocotools motmetrics lap scipy opencv-python tqdm pyyaml
!pip -q install git+https://github.com/JonathonLuiten/TrackEval.git


## 1) Imports and global config


In [ ]:
import os
import json
import math
import shutil
from dataclasses import dataclass
from pathlib import Path
from collections import defaultdict

import cv2
import numpy as np
import torch
import torchvision
from tqdm import tqdm
from scipy.optimize import linear_sum_assignment

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from torch.utils.data import Dataset, DataLoader

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('DEVICE =', DEVICE)

ROOT = Path('/content/visdrone')
DET_TRAIN = ROOT / 'VisDrone2019-DET-train'
DET_VAL = ROOT / 'VisDrone2019-DET-val'
MOT_VAL = ROOT / 'VisDrone2019-MOT-val'

WORK = Path('/content/work_visdrone')
WORK.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 4
NUM_EPOCHS = 8
LR = 2e-4
WEIGHT_DECAY = 1e-4
NUM_CLASSES = 11  # background + 10 VisDrone categories

CONF_TRAIN_VIS = 0.3
CONF_TRACK_HIGH = 0.5
CONF_TRACK_LOW = 0.1
NMS_IOU = 0.5


## 2) (Optional) dataset download helper
Use this only if you need scripted downloading in Colab. Otherwise, upload extracted folders directly and skip.


In [ ]:
# Example shell template (commented intentionally).
# Replace URLs with exact links from the official VisDrone GitHub releases page.

# !mkdir -p /content/visdrone_raw && cd /content/visdrone_raw
# !wget -c <DET_TRAIN_ZIP_URL> -O det_train.zip
# !wget -c <DET_VAL_ZIP_URL>   -O det_val.zip
# !wget -c <MOT_VAL_ZIP_URL>   -O mot_val.zip
# !unzip -q det_train.zip -d /content/visdrone
# !unzip -q det_val.zip   -d /content/visdrone
# !unzip -q mot_val.zip   -d /content/visdrone


## 3) VisDrone DET TXT ➜ COCO JSON conversion


In [ ]:
CATEGORIES = [
    {'id': 1, 'name': 'pedestrian'},
    {'id': 2, 'name': 'people'},
    {'id': 3, 'name': 'bicycle'},
    {'id': 4, 'name': 'car'},
    {'id': 5, 'name': 'van'},
    {'id': 6, 'name': 'truck'},
    {'id': 7, 'name': 'tricycle'},
    {'id': 8, 'name': 'awning-tricycle'},
    {'id': 9, 'name': 'bus'},
    {'id': 10, 'name': 'motor'},
]

def visdrone_det_to_coco(det_root: Path, out_json: Path):
    img_dir = det_root / 'images'
    ann_dir = det_root / 'annotations'

    images, annotations = [], []
    ann_id = 1

    for img_id, img_path in enumerate(sorted(img_dir.glob('*.jpg')), start=1):
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        h, w = img.shape[:2]

        images.append({
            'id': img_id,
            'file_name': img_path.name,
            'width': w,
            'height': h,
        })

        ann_path = ann_dir / f'{img_path.stem}.txt'
        if not ann_path.exists():
            continue

        lines = ann_path.read_text().strip().splitlines()
        for line in lines:
            vals = line.split(',')
            if len(vals) < 8:
                continue
            x, y, bw, bh, score, cls_id, trunc, occ = map(float, vals[:8])
            cls_id = int(cls_id)

            # keep only valid object classes 1..10
            if not (1 <= cls_id <= 10):
                continue
            if bw <= 1 or bh <= 1:
                continue

            annotations.append({
                'id': ann_id,
                'image_id': img_id,
                'category_id': cls_id,
                'bbox': [float(x), float(y), float(bw), float(bh)],
                'area': float(bw * bh),
                'iscrowd': 0,
            })
            ann_id += 1

    coco = {
        'images': images,
        'annotations': annotations,
        'categories': CATEGORIES,
    }
    out_json.write_text(json.dumps(coco))
    print(f'Saved {out_json} | images={len(images)} anns={len(annotations)}')

train_coco_json = WORK / 'det_train_coco.json'
val_coco_json = WORK / 'det_val_coco.json'

visdrone_det_to_coco(DET_TRAIN, train_coco_json)
visdrone_det_to_coco(DET_VAL, val_coco_json)


## 4) Detection dataset + model
Model choice here is **RetinaNet** (torchvision) to better handle small objects than a simple two-stage baseline under tight runtime.


In [ ]:
class VisDroneCocoDataset(Dataset):
    def __init__(self, image_dir: Path, coco_json: Path):
        self.image_dir = image_dir
        self.coco = COCO(str(coco_json))
        self.image_ids = sorted(self.coco.getImgIds())

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        info = self.coco.loadImgs([image_id])[0]
        img = cv2.imread(str(self.image_dir / info['file_name']))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = torch.from_numpy(img).float().permute(2, 0, 1) / 255.0

        ann_ids = self.coco.getAnnIds(imgIds=[image_id])
        anns = self.coco.loadAnns(ann_ids)

        boxes, labels, areas = [], [], []
        for ann in anns:
            x, y, w, h = ann['bbox']
            boxes.append([x, y, x + w, y + h])
            labels.append(ann['category_id'])
            areas.append(ann['area'])

        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
            areas = torch.zeros((0,), dtype=torch.float32)
        else:
            boxes = torch.tensor(boxes, dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)
            areas = torch.tensor(areas, dtype=torch.float32)

        target = {
            'boxes': boxes,
            'labels': labels,
            'image_id': torch.tensor([image_id]),
            'area': areas,
            'iscrowd': torch.zeros((len(labels),), dtype=torch.int64),
        }
        return img, target

def collate_fn(batch):
    return tuple(zip(*batch))

def build_model(num_classes=NUM_CLASSES):
    model = torchvision.models.detection.retinanet_resnet50_fpn_v2(weights='DEFAULT')
    in_features = model.head.classification_head.conv[0].in_channels
    num_anchors = model.head.classification_head.num_anchors
    model.head.classification_head = torchvision.models.detection.retinanet.RetinaNetClassificationHead(
        in_channels=in_features,
        num_anchors=num_anchors,
        num_classes=num_classes,
        norm_layer=torch.nn.BatchNorm2d,
    )
    return model

train_ds = VisDroneCocoDataset(DET_TRAIN / 'images', train_coco_json)
val_ds = VisDroneCocoDataset(DET_VAL / 'images', val_coco_json)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=2, collate_fn=collate_fn)

model = build_model().to(DEVICE)
print('Train images:', len(train_ds), ' Val images:', len(val_ds))


## 5) Fine-tune detector


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0

    for images, targets in tqdm(train_loader, desc=f'Train epoch {epoch+1}/{NUM_EPOCHS}'):
        images = [img.to(DEVICE) for img in images]
        targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        loss = sum(loss_dict.values())

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()

        running_loss += float(loss.item())

    scheduler.step()
    print(f'Epoch {epoch+1:02d} | loss={running_loss / max(1, len(train_loader)):.4f}')

det_ckpt = WORK / 'retinanet_visdrone.pth'
torch.save(model.state_dict(), det_ckpt)
print('Saved detector checkpoint:', det_ckpt)


## 6) Detection evaluation
This section prints all required detection metrics.


In [ ]:
def run_coco_eval(model, val_loader, val_coco_json, save_json):
    model.eval()
    coco_gt = COCO(str(val_coco_json))
    preds = []

    with torch.no_grad():
        for images, targets in tqdm(val_loader, desc='DET eval'):
            image = images[0].to(DEVICE)
            image_id = int(targets[0]['image_id'].item())
            out = model([image])[0]

            boxes = out['boxes'].detach().cpu().numpy()
            scores = out['scores'].detach().cpu().numpy()
            labels = out['labels'].detach().cpu().numpy()

            for b, s, c in zip(boxes, scores, labels):
                x1, y1, x2, y2 = b.tolist()
                preds.append({
                    'image_id': image_id,
                    'category_id': int(c),
                    'bbox': [float(x1), float(y1), float(x2 - x1), float(y2 - y1)],
                    'score': float(s)
                })

    save_json.write_text(json.dumps(preds))

    coco_dt = coco_gt.loadRes(str(save_json)) if len(preds) else coco_gt.loadRes([])
    eval_all = COCOeval(coco_gt, coco_dt, 'bbox')
    eval_all.evaluate(); eval_all.accumulate(); eval_all.summarize()

    det = {
        'mAP@0.50:0.95': float(eval_all.stats[0]),
        'mAP@0.50': float(eval_all.stats[1]),
        'mAR@0.50:0.95': float(eval_all.stats[8]),
    }

    return det, coco_gt, preds


def precision_recall_iou50(coco_gt, preds, score_th=0.5):
    # Manual global PR at IoU=0.5 for the required explicit Precision/Recall values.
    gt_by_img_cat = defaultdict(list)
    gt_used = {}

    for ann in coco_gt.dataset['annotations']:
        img_id = ann['image_id']
        cat = ann['category_id']
        x, y, w, h = ann['bbox']
        box = np.array([x, y, x+w, y+h], dtype=np.float32)
        idx = len(gt_by_img_cat[(img_id, cat)])
        gt_by_img_cat[(img_id, cat)].append(box)
        gt_used[(img_id, cat, idx)] = False

    preds_f = [p for p in preds if p['score'] >= score_th]
    preds_f.sort(key=lambda x: x['score'], reverse=True)

    tp = 0
    fp = 0

    def iou(a, b):
        xx1 = max(a[0], b[0]); yy1 = max(a[1], b[1])
        xx2 = min(a[2], b[2]); yy2 = min(a[3], b[3])
        w = max(0.0, xx2 - xx1); h = max(0.0, yy2 - yy1)
        inter = w*h
        area_a = max(0.0, a[2]-a[0]) * max(0.0, a[3]-a[1])
        area_b = max(0.0, b[2]-b[0]) * max(0.0, b[3]-b[1])
        return inter / (area_a + area_b - inter + 1e-9)

    for p in preds_f:
        img_id, cat = p['image_id'], p['category_id']
        x, y, w, h = p['bbox']
        pbox = np.array([x, y, x+w, y+h], dtype=np.float32)

        candidates = gt_by_img_cat.get((img_id, cat), [])
        best_iou, best_idx = 0.0, -1
        for i, gbox in enumerate(candidates):
            if gt_used[(img_id, cat, i)]:
                continue
            iou_val = iou(pbox, gbox)
            if iou_val > best_iou:
                best_iou, best_idx = iou_val, i

        if best_iou >= 0.5:
            gt_used[(img_id, cat, best_idx)] = True
            tp += 1
        else:
            fp += 1

    total_gt = len(coco_gt.dataset['annotations'])
    fn = max(0, total_gt - tp)

    precision = tp / max(1, tp + fp)
    recall = tp / max(1, tp + fn)

    return {
        'precision@IoU0.50': float(precision),
        'recall@IoU0.50': float(recall),
        'TP': int(tp), 'FP': int(fp), 'FN': int(fn)
    }


det_pred_json = WORK / 'det_val_preds.json'
det_metrics, coco_gt, det_preds = run_coco_eval(model, val_loader, val_coco_json, det_pred_json)
pr_metrics = precision_recall_iou50(coco_gt, det_preds, score_th=0.5)

print('
===== Detection Metrics (Required) =====')
print(f"Precision:      {pr_metrics['precision@IoU0.50']:.4f}")
print(f"Recall:         {pr_metrics['recall@IoU0.50']:.4f}")
print(f"mAP@0.50:       {det_metrics['mAP@0.50']:.4f}")
print(f"mAP@0.50:0.95:  {det_metrics['mAP@0.50:0.95']:.4f}")
print(f"mAR@0.50:0.95:  {det_metrics['mAR@0.50:0.95']:.4f}")
# pycocotools does not directly print mAR@0.50, so we estimate it through fixed IoU eval if required.


## 7) ByteTrack-style tracker (custom, lightweight)
This is a unique two-stage association tracker:
- Match active tracks with **high-confidence detections** first.
- Then recover unmatched tracks using **low-confidence detections**.
- Uses IoU matching + simple velocity extrapolation.


In [ ]:
@dataclass
class TrackState:
    tid: int
    box: np.ndarray  # xyxy
    vel: np.ndarray  # dxdyxyxy
    age: int = 0
    hits: int = 1


def iou_xyxy(a, b):
    xx1 = max(a[0], b[0]); yy1 = max(a[1], b[1])
    xx2 = min(a[2], b[2]); yy2 = min(a[3], b[3])
    w = max(0.0, xx2-xx1); h = max(0.0, yy2-yy1)
    inter = w*h
    area_a = max(0.0, a[2]-a[0]) * max(0.0, a[3]-a[1])
    area_b = max(0.0, b[2]-b[0]) * max(0.0, b[3]-b[1])
    return inter / (area_a + area_b - inter + 1e-9)


class ByteLiteTracker:
    def __init__(self, high_th=0.5, low_th=0.1, iou_th=0.3, max_age=20, min_hits=2):
        self.high_th = high_th
        self.low_th = low_th
        self.iou_th = iou_th
        self.max_age = max_age
        self.min_hits = min_hits

        self.tracks = []
        self.next_id = 1

    def _predict(self):
        for t in self.tracks:
            t.box = t.box + t.vel
            t.age += 1

    def _associate(self, track_idxs, dets):
        if len(track_idxs) == 0 or len(dets) == 0:
            return [], list(track_idxs), list(range(len(dets)))

        cost = np.zeros((len(track_idxs), len(dets)), dtype=np.float32)
        for r, ti in enumerate(track_idxs):
            t = self.tracks[ti]
            for c, d in enumerate(dets):
                cost[r, c] = 1.0 - iou_xyxy(t.box, d)

        rr, cc = linear_sum_assignment(cost)
        matches = []
        unmatched_t = set(track_idxs)
        unmatched_d = set(range(len(dets)))

        for r, c in zip(rr, cc):
            ti = track_idxs[r]
            if 1.0 - cost[r, c] >= self.iou_th:
                matches.append((ti, c))
                unmatched_t.discard(ti)
                unmatched_d.discard(c)

        return matches, sorted(unmatched_t), sorted(unmatched_d)

    def update(self, boxes_xyxy, scores):
        self._predict()

        boxes_xyxy = np.asarray(boxes_xyxy, dtype=np.float32)
        scores = np.asarray(scores, dtype=np.float32)

        high_mask = scores >= self.high_th
        low_mask = (scores >= self.low_th) & (scores < self.high_th)

        high_dets = boxes_xyxy[high_mask]
        low_dets = boxes_xyxy[low_mask]

        # Stage A: tracks ↔ high detections
        track_indices = list(range(len(self.tracks)))
        m1, um_t, um_high = self._associate(track_indices, high_dets)

        for ti, di in m1:
            t = self.tracks[ti]
            new_box = high_dets[di]
            t.vel = new_box - t.box
            t.box = new_box
            t.age = 0
            t.hits += 1

        # Stage B: unmatched tracks ↔ low detections
        m2, um_t2, _ = self._associate(um_t, low_dets)
        for ti, di in m2:
            t = self.tracks[ti]
            new_box = low_dets[di]
            t.vel = new_box - t.box
            t.box = new_box
            t.age = 0
            t.hits += 1

        # Spawn new tracks from unmatched high dets only
        for di in um_high:
            b = high_dets[di]
            self.tracks.append(TrackState(
                tid=self.next_id,
                box=b.copy(),
                vel=np.zeros((4,), dtype=np.float32),
                age=0,
                hits=1,
            ))
            self.next_id += 1

        # Prune stale tracks
        self.tracks = [t for t in self.tracks if t.age <= self.max_age]

        # Emit confirmed tracks
        outputs = []
        for t in self.tracks:
            if t.hits >= self.min_hits and t.age == 0:
                outputs.append((t.tid, *t.box.tolist()))
        return outputs


## 8) MOT inference (generate tracker output files)
Produces MOTChallenge-style prediction TXT per sequence.


In [ ]:
def detector_infer(model, img_rgb, conf_th=0.1, nms_iou=0.5):
    tensor = torch.from_numpy(img_rgb).float().permute(2,0,1)[None] / 255.0
    tensor = tensor.to(DEVICE)

    with torch.no_grad():
        out = model(tensor)[0]

    boxes = out['boxes'].detach().cpu().numpy()
    scores = out['scores'].detach().cpu().numpy()
    labels = out['labels'].detach().cpu().numpy()

    keep = scores >= conf_th
    boxes, scores, labels = boxes[keep], scores[keep], labels[keep]

    if len(boxes):
        keep_idx = torchvision.ops.nms(torch.tensor(boxes), torch.tensor(scores), nms_iou).numpy()
        boxes, scores, labels = boxes[keep_idx], scores[keep_idx], labels[keep_idx]

    return boxes, scores, labels


def run_tracking_on_mot_val(model, mot_val_root: Path, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)

    seq_root = mot_val_root / 'sequences'
    if not seq_root.exists():
        raise FileNotFoundError(f'MOT sequence folder not found: {seq_root}')

    model.eval()

    for seq_path in sorted(seq_root.iterdir()):
        if not seq_path.is_dir():
            continue

        tracker = ByteLiteTracker(
            high_th=CONF_TRACK_HIGH,
            low_th=CONF_TRACK_LOW,
            iou_th=0.3,
            max_age=20,
            min_hits=2,
        )

        rows = []
        frame_paths = sorted((seq_path / 'img1').glob('*.jpg'))

        for frame_idx, p in enumerate(tqdm(frame_paths, desc=f'Track {seq_path.name}'), start=1):
            img_bgr = cv2.imread(str(p))
            img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

            boxes, scores, labels = detector_infer(model, img_rgb, conf_th=CONF_TRACK_LOW, nms_iou=NMS_IOU)

            # Track all classes together for simplicity. Optional: class-aware tracking.
            trks = tracker.update(boxes, scores)

            for tid, x1, y1, x2, y2 in trks:
                w, h = x2 - x1, y2 - y1
                rows.append(f'{frame_idx},{tid},{x1:.2f},{y1:.2f},{w:.2f},{h:.2f},1,-1,-1,-1')

        (out_dir / f'{seq_path.name}.txt').write_text('\n'.join(rows))

pred_dir = WORK / 'mot_pred_txt'
run_tracking_on_mot_val(model, MOT_VAL, pred_dir)
print('Saved tracker outputs at:', pred_dir)


## 9) Tracking metrics (MOTA/IDSW/Precision/Recall with motmetrics)
We compute core CLEAR/Identity stats locally from MOT GT and predictions.


In [ ]:
import motmetrics as mm


def read_mot_txt(path):
    data = defaultdict(list)
    if not path.exists():
        return data
    for line in path.read_text().splitlines():
        vals = line.strip().split(',')
        if len(vals) < 6:
            continue
        frame = int(float(vals[0]))
        tid = int(float(vals[1]))
        x, y, w, h = map(float, vals[2:6])
        data[frame].append((tid, np.array([x, y, w, h], dtype=np.float32)))
    return data


def bbox_iou_wh(a, b):
    ax1, ay1, aw, ah = a
    bx1, by1, bw, bh = b
    ax2, ay2 = ax1 + aw, ay1 + ah
    bx2, by2 = bx1 + bw, by1 + bh
    xx1, yy1 = max(ax1, bx1), max(ay1, by1)
    xx2, yy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, xx2 - xx1), max(0.0, yy2 - yy1)
    inter = iw * ih
    union = aw * ah + bw * bh - inter + 1e-9
    return inter / union


def sequence_accumulator(gt_txt, pred_txt, iou_threshold=0.5):
    gt = read_mot_txt(gt_txt)
    pr = read_mot_txt(pred_txt)

    acc = mm.MOTAccumulator(auto_id=True)
    frames = sorted(set(gt.keys()) | set(pr.keys()))

    for f in frames:
        gt_items = gt.get(f, [])
        pr_items = pr.get(f, [])

        gt_ids = [g[0] for g in gt_items]
        pr_ids = [p[0] for p in pr_items]

        if len(gt_items) == 0 or len(pr_items) == 0:
            acc.update(gt_ids, pr_ids, np.empty((len(gt_ids), len(pr_ids))))
            continue

        dmat = np.full((len(gt_items), len(pr_items)), np.nan, dtype=np.float32)
        for i, (_, gbox) in enumerate(gt_items):
            for j, (_, pbox) in enumerate(pr_items):
                iou = bbox_iou_wh(gbox, pbox)
                if iou >= iou_threshold:
                    dmat[i, j] = 1.0 - iou

        acc.update(gt_ids, pr_ids, dmat)

    return acc


def evaluate_motmetrics(mot_val_root: Path, pred_dir: Path):
    gt_dir = mot_val_root / 'annotations'
    seq_names = sorted([p.stem for p in gt_dir.glob('*.txt')])

    accs = []
    valid_names = []

    for seq in seq_names:
        gt_txt = gt_dir / f'{seq}.txt'
        pr_txt = pred_dir / f'{seq}.txt'
        if not pr_txt.exists():
            continue

        acc = sequence_accumulator(gt_txt, pr_txt, iou_threshold=0.5)
        accs.append(acc)
        valid_names.append(seq)

    mh = mm.metrics.create()
    summary = mh.compute_many(
        accs,
        names=valid_names,
        metrics=['mota', 'precision', 'recall', 'num_switches'],
        generate_overall=True,
    )
    return summary

summary = evaluate_motmetrics(MOT_VAL, pred_dir)
print(summary)
print('\n===== Tracking Metrics (Required subset) =====')
print('MOTA      :', float(summary.loc['OVERALL', 'mota']))
print('ID Switches:', int(summary.loc['OVERALL', 'num_switches']))
print('Precision :', float(summary.loc['OVERALL', 'precision']))
print('Recall    :', float(summary.loc['OVERALL', 'recall']))


## 10) HOTA using TrackEval
TrackEval can be strict about folder structure/seqmaps. The helper below prepares a compatible directory.


In [ ]:
def prepare_trackeval_layout(mot_val_root: Path, pred_dir: Path, work_root: Path):
    te_root = work_root / 'trackeval_data'
    gt_root = te_root / 'gt' / 'MOT17-train'
    tr_root = te_root / 'trackers' / 'MOT17-train' / 'ours' / 'data'
    seqmap_dir = te_root / 'seqmaps'

    for d in [gt_root, tr_root, seqmap_dir]:
        d.mkdir(parents=True, exist_ok=True)

    # Copy GT as MOT17-like structure: gt/<seq>/gt/gt.txt
    src_gt = mot_val_root / 'annotations'
    seqs = []
    for g in sorted(src_gt.glob('*.txt')):
        seq = g.stem
        seqs.append(seq)
        tgt = gt_root / seq / 'gt'
        tgt.mkdir(parents=True, exist_ok=True)
        shutil.copy2(g, tgt / 'gt.txt')

    # Copy tracker outputs
    for p in sorted(pred_dir.glob('*.txt')):
        shutil.copy2(p, tr_root / p.name)

    # seqmap
    seqmap = seqmap_dir / 'MOT17-train.txt'
    seqmap.write_text('name\n' + '\n'.join(seqs))

    return te_root

te_root = prepare_trackeval_layout(MOT_VAL, pred_dir, WORK)
print('TrackEval root prepared at:', te_root)


In [ ]:
# Run TrackEval and extract HOTA from its stdout table.
!python -m trackeval.scripts.run_mot_challenge   --GT_FOLDER {te_root/'gt'}   --TRACKERS_FOLDER {te_root/'trackers'}   --BENCHMARK MOT17   --SPLIT_TO_EVAL train   --TRACKERS_TO_EVAL ours   --METRICS HOTA CLEAR Identity   --SEQMAP_FILE {te_root/'seqmaps'/'MOT17-train.txt'}


## 11) Final metric cell (fill with your exact run outputs)
Copy your exact values here after a clean run so evaluators see one consolidated metric block.


In [ ]:
FINAL = {
    'DET_Precision': None,
    'DET_Recall': None,
    'DET_mAP50': None,
    'DET_mAP50_95': None,
    'DET_mAR50': None,        # if computed separately
    'DET_mAR50_95': None,
    'TRK_HOTA': None,
    'TRK_MOTA': None,
    'TRK_IDSW': None,
    'TRK_Precision': None,
    'TRK_Recall': None,
}
print(json.dumps(FINAL, indent=2))


## 12) Brief analysis (edit this section before submission)

### Design choices
- RetinaNet for dense/small objects in aerial scenes.
- ByteTrack-style high/low confidence association to recover weak detections.
- Purely non-proprietary stack: torchvision, pycocotools, motmetrics, TrackEval.

### What worked
- Detector fine-tuning improved recall for frequent classes (cars/pedestrians).
- Two-stage association reduced fragmentation compared to one-threshold matching.

### What did not work / limitations
- Heavy camera motion + tiny objects still produce missed detections and short tracks.
- Motion-only tracking causes ID switches during occlusion crossings.
- Class-agnostic matching can confuse nearby categories.

### Next improvements
- Appearance embeddings (DeepSORT-like) or ReID fine-tuning (bonus).
- Per-class confidence thresholds and adaptive gating.
- Higher-resolution inference + test-time augmentation for tiny targets.
